# Data Warehouse ETL and OLAP

## Sugarcane Agricultural Analytics

**Purpose:** Convert the cleaned Master Dataset into a small analytical data warehouse using ETL operations, a Star Schema, surrogate keys, validation checks, and OLAP-style analysis.

### Existing work reused
- `01_Data_Inspection.ipynb` → source inspection, standardization, integration and Master Dataset creation.
- `02_Data_Analysis_and_EDA.ipynb` → missing-value analysis, EDA, nutrient/weather/management analysis and correlation analysis.

### MDA tasks addressed in this notebook
4. Design Data Warehouse Architecture  
5. Design Dimensional Model  
6. Extract Source Data  
7. Perform Data Cleaning  
8. Perform Data Transformation  
9. Create Dimension Tables  
10. Create & Load Fact Table  
11. Validate ETL  
12. Create Analytical Model / OLAP structure  
13. Implement OLAP Operations  
14. Prepare data for Interactive Dashboard  
15. Document and present the workflow


In [6]:
# ============================================================
# 1. IMPORTS AND PROJECT PATHS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

# Notebook is expected to be inside the project's notebooks/ folder.
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
WAREHOUSE_DIR = OUTPUT_DIR / "data_warehouse"

WAREHOUSE_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH = OUTPUT_DIR / "tables" / "Sugarcane_Master_Dataset.csv"
DS18_PATH = DATA_DIR / "DS18_Rainfall_TimeSeries_West_Champaran.csv"

print("Project root:", PROJECT_ROOT)
print("Master dataset:", MASTER_PATH)
print("DS18 rainfall:", DS18_PATH)
print("Warehouse output:", WAREHOUSE_DIR)


Project root: C:\Users\Arindam Dwivedi\.vscode\sugarcane-agricultural-analytics
Master dataset: C:\Users\Arindam Dwivedi\.vscode\sugarcane-agricultural-analytics\outputs\tables\Sugarcane_Master_Dataset.csv
DS18 rainfall: C:\Users\Arindam Dwivedi\.vscode\sugarcane-agricultural-analytics\data\DS18_Rainfall_TimeSeries_West_Champaran.csv
Warehouse output: C:\Users\Arindam Dwivedi\.vscode\sugarcane-agricultural-analytics\outputs\data_warehouse


## 2. Business Requirements

The warehouse is designed to support analysis of:

- sugarcane yield and sugar quality,
- nutrient status and fertilizer-related measures,
- vegetation and chlorophyll indicators,
- precipitation and weather/stress indicators,
- crop, location and management factors,
- seasonal comparisons,
- plot-level spatial analysis,
- OLAP-style aggregation for dashboard reporting.

### Core analytical KPIs

- Total / average TCH
- Total / average TSH
- Average CCS
- Average sugar recovery
- Average rainfall
- Average nutrient measurements
- Yield by season
- Yield by cultivar
- Yield by nutrient class
- Yield by location


## 3. Data Warehouse Architecture

The project follows the architecture:

**Source Excel/CSV files → Staging → ETL → Data Warehouse → Analytical/OLAP Model → Power BI Dashboard**

The eight original agricultural datasets were already integrated in Notebook 1. This notebook adds DS18, a 2021–2024 ERA5-Land monthly rainfall series, and transforms it to the warehouse's Plot × Season grain.

### Fact-table grain

> **One row in `Fact_Agriculture` represents one Plot × Season agricultural observation.**

DS18 contains monthly regional rainfall. It is therefore aggregated to the agricultural season/year before being attached to the fact table. This prevents monthly rainfall rows from multiplying the agricultural fact rows.


## 4. Star Schema

### Dimension tables

1. `Dim_Plot`
2. `Dim_Location`
3. `Dim_Crop`
4. `Dim_Date`
5. `Dim_Nutrient`
6. `Dim_Vegetation`
7. `Dim_Weather`

### Fact table

- `Fact_Agriculture`

Conceptually:

```text
                         Dim_Date
                            |
                            |
 Dim_Plot -------- Fact_Agriculture -------- Dim_Crop
                            |
 Dim_Location -------------|------------- Dim_Nutrient
                            |
                     Dim_Vegetation
                            |
                       Dim_Weather
```

Surrogate integer keys are generated for dimensions and stored as foreign keys in the fact table.


In [7]:
# ============================================================
# 5. EXTRACT - LOAD THE CLEAN MASTER DATASET
# ============================================================

if not MASTER_PATH.exists():
    raise FileNotFoundError(
        f"Master dataset not found at {MASTER_PATH}. "
        "Run 01_Data_Inspection.ipynb / 02_Data_Analysis_and_EDA.ipynb first."
    )

master = pd.read_csv(MASTER_PATH)

print("Master dataset shape:", master.shape)
print("Unique Plot_IDs:", master["Plot_ID"].nunique())
display(master.head(3))


Master dataset shape: (100, 126)
Unique Plot_IDs: 100


,Plot_ID,Village_registry,Block_registry,Season_registry,Latitude_deg,Longitude_deg,Plot_Area_ha,Irrigation_type,Soil_type,Planting_date,Harvest_date,Growth_days,Notes,Village_tsh,Season_tsh,Cultivar,Cycle,Block_tsh,TCH_t_per_ha,CCS_pct,TSH_t_per_ha,Sugar_recovery_pct,Juice_purity_pct,Brix_pct,Pol_pct,Mill_ID,Village,Season,Source,Grid_lat,Grid_lon,Precip_growth_phase_mm,Precip_Jun_Aug_mm,Precip_Sep_Nov_mm,Precip_Dec_Feb_mm,Dry_spell_days_gt5,Wet_days_gt10mm,Max_1day_precip_mm,Drought_flag,Excess_rain_flag,SPI_3month,Village_N,Season_N,Sample date,Growth stage,Leaf position,N content pct DW,N class,Kjeldahl N mg kg,Total protein pct,Chlorophyll SPAD,N fertiliser kg ha,Method,Lab ID,Village_P,Season_P,Sample date_P,Growth stage_P,P content pct DW,P class,P mg per kg DW,P2O5 pct,Available P Olsen,P fertiliser kg ha,P uptake kg ha,Method_P,Lab ID_P,Village_K,Season_K,Sample date_K,Growth stage_K,K content pct DW,K class,K mg per kg DW,K2O pct,Exchangeable K meq,K fertiliser kg ha,K Na ratio,Method_K,Lab ID_K,Village_GNDVI,Season_GNDVI,Acquisition date,Growth stage_GNDVI,GNDVI,GNDVI class,NDVI,EVI,LSWI,Canopy LAI est,Chlorophyll est ug cm2,Formula,Source dataset,Village_CHL,Cultivar_CHL,Cycle_CHL,Season_CHL,Block,Sample date_CHL,Time of sampling,Growth stage_CHL,Leaf position_CHL,Leaf age days,Chl a mg per gFW,Chl b mg per gFW,Total Chl mg per gFW,Chl a to b ratio,Carotenoid mg per gFW,Chl to Carotenoid ratio,SPAD value,Chlorophyll Index CI,Pheophytin mg per gFW,Chl degradation pct,Fv Fm PSII efficiency,Phi PSII quantum yield,NPQ non photochem quenching,Chl specific absorption,Light use efficiency LUE,GNDVI from DS14,N content pct from DS11,Nutrient balance NB DS16,Stress type,Chl status class,Extraction method,Lab ID_CHL,Notes_CHL
0,WC-001,Matiaria,2,2021-22,26.87983,84.67516,0.400,Canal,Sandy_loam,2021-11-01,2022-10-15,344,NaN,Matiaria,2021-22,CoVSI9,Plant,2,11.18,11.80,1.023,7.71,84.18,19.36,20.64,WC_Mill_03,Matiaria,2021-22,ERA5_0.25deg,27.01,84.64,1066.3,650.0,286.3,47.4,10,69,45.2,No,No,-0.465,Matiaria,2021-22,2022-06-01,Grand_growth,4th_leaf,2.896,Excess,28960.0,18.100,54.1,150.0,Kjeldahl_digestion,ICAR-LEAF-N-0001,Matiaria,2021-22,2022-06-01,Grand_growth,0.2241,Adequate,2241.0,0.5134,19.56,50.0,13.44,Vanadomolybdate_colorimetry,ICAR-LEAF-P-0001,Matiaria,2021-22,2022-06-01,Grand_growth,1.482,Adequate,14820.0,1.786,0.941,60.0,11.98,Flame_photometry,ICAR-LEAF-K-0001,Matiaria,2021-22,2022-01-10,Tillering,0.6253,High_vigour,0.8474,1.4089,0.1474,3.34,31.53,GNDVI=(NIR-Green)/(NIR+Green),DS4_Landsat_Bands,Matiaria,CoVSI9,Plant,2021-22,2.0,2022-06-01,17:00,Grand_growth,4th_leaf,29.0,5.0000,1.5979,6.5979,3.129,1.3081,5.044,72.00,0.900,0.4744,7.19,0.7545,0.6786,0.979,0.12487,0.01773,0.62527,2.896,0.53153,Pest,High,Ethanol_96pct,ICAR-CHL-DS17-0001,NaN
1,WC-002,Sirsa,4,2023-24,27.03752,84.74989,1.435,Tubewell,Clay_loam,2021-11-04,2022-10-18,388,NaN,Sirsa,2023-24,Co0238,Ratoon1,4,104.19,11.42,9.136,7.76,86.65,20.42,20.91,WC_Mill_03,Sirsa,2023-24,IMD_gridded,26.81,84.46,958.6,609.9,271.3,57.3,4,76,168.5,No,No,0.612,Sirsa,2023-24,2022-06-04,Grand_growth,4th_leaf,1.499,Deficient,14990.0,9.369,28.8,180.0,Kjeldahl_digestion,ICAR-LEAF-N-0002,Sirsa,2023-24,2022-06-04,Grand_growth,0.2157,Adequate,2157.0,0.4942,30.41,40.0,9.77,Vanadomolybdate_colorimetry,ICAR-LEAF-P-0002,Sirsa,2023-24,2022-06-04,Grand_growth,1.290,Adequate,12900.0,1.554,0.776,80.0,8.59,Flame_photometry,ICAR-LEAF-K-0002,Sirsa,2023-24,2022-01-12,Grand_growth,0.8444,High_vigour,0.7426,0.6684,0.6517,5.06,38.99,GNDVI=(NIR-Green)/(NIR+Green),DS4_Landsat_Bands,Sirsa,Co0238,Ratoon1,2023-24,4.0,2022-06-04,11:00,Grand_growth,4th_leaf,20.0,4.9785,2.0157,6.9942,2.470,1.5325,4.564,56.96,0.712,0.5331,7.62,0.8500,0.8109,1.623,0.13028,0.01752,0.84435,1.499,0.39002,Nutrient_deficiency,High,Acetone_80pct,ICAR-CHL-DS17-0002,NaN
2,WC-003,Matiaria,3,2023-24,27.03765,84.74865,1.550,Canal,Sandy_loam,2021-11-07,2022-10-21,355,NaN,Matiaria,2023-24,CoS767,Ratoon2,3,71.81,10.75,6.720,7.83,86.83,21.26,16.7

In [8]:
# ============================================================
# 6. EXTRACT - LOAD DS18 MONTHLY RAINFALL
# ============================================================

if not DS18_PATH.exists():
    print("DS18 was not found inside the project's data/ folder.")
    print("Copy DS18_Rainfall_TimeSeries_West_Champaran.csv into:", DATA_DIR)
    ds18 = pd.DataFrame()
else:
    ds18 = pd.read_csv(DS18_PATH)
    ds18["Date"] = pd.to_datetime(ds18["Date"], errors="coerce")
    ds18["Year"] = pd.to_numeric(ds18["Year"], errors="coerce").astype("Int64")
    ds18["Month"] = pd.to_numeric(ds18["Month"], errors="coerce").astype("Int64")
    ds18["Rainfall_mm"] = pd.to_numeric(ds18["Rainfall_mm"], errors="coerce")

    print("DS18 shape:", ds18.shape)
    print("Date range:", ds18["Date"].min(), "to", ds18["Date"].max())
    display(ds18.head())


DS18 shape: (48, 8)
Date range: 2021-01-01 00:00:00 to 2024-12-01 00:00:00


,Date,Year,Month,Month_Name,Rainfall_mm,Location,Source,Spatial_Method
0,2021-01-01,2021,1,January,1.23,"West Champaran, Bihar",ERA5-Land,Mean of 0.1-degree grid cells in selected bounding box
1,2021-02-01,2021,2,February,3.58,"West Champaran, Bihar",ERA5-Land,Mean of 0.1-degree grid cells in selected bounding box
2,2021-03-01,2021,3,March,2.67,"West Champaran, Bihar",ERA5-Land,Mean of 0.1-degree grid cells in selected bounding box
3,2021-04-01,2021,4,April,8.84,"West Champaran, Bihar",ERA5-Land,Mean of 0.1-degree grid cells in selected bounding box
4,2021-05-01,2021,5,May,408.73,"West Champaran, Bihar",ERA5-Land,Mean of 0.1-degree grid cells in selected bounding box


### DS18 integration assumption

The existing Master Dataset uses agricultural seasons such as `2021-22`, `2022-23`, and `2023-24`, while DS18 is a calendar-month rainfall series.

For this warehouse demonstration, the **first year in the agricultural season** is used to link DS18 to the season. For example:

- `2021-22` → DS18 year 2021
- `2022-23` → DS18 year 2022
- `2023-24` → DS18 year 2023

This is a documented integration assumption, not a claim that the ERA5-Land annual calendar year exactly equals the crop's full physiological rainfall exposure.


In [9]:
# ============================================================
# 7. TRANSFORM DS18 TO SEASON-LEVEL RAINFALL
# ============================================================

def season_start_year(value):
    match = re.search(r"(20\d{2})", str(value))
    return int(match.group(1)) if match else np.nan

if not ds18.empty:
    rainfall_year = (
        ds18.groupby("Year", as_index=False)["Rainfall_mm"]
        .sum()
        .rename(columns={"Year": "Season_Start_Year",
                         "Rainfall_mm": "Annual_Rainfall_mm"})
    )

    # Agricultural-period groupings using calendar months.
    monthly = ds18.copy()

    period = pd.Series(np.nan, index=monthly.index, dtype="object")
    period[monthly["Month"].between(6, 8)] = "Jun_Aug"
    period[monthly["Month"].between(9, 11)] = "Sep_Nov"
    period[(monthly["Month"] == 12) | (monthly["Month"].between(1, 2))] = "Dec_Feb"
    period[monthly["Month"].between(3, 5)] = "Growth_phase"
    monthly["Rainfall_Period"] = period

    rainfall_periods = (
        monthly.pivot_table(
            index="Year",
            columns="Rainfall_Period",
            values="Rainfall_mm",
            aggfunc="sum"
        )
        .reset_index()
        .rename(columns={
            "Year": "Season_Start_Year",
            "Growth_phase": "Precip_growth_phase_mm",
            "Jun_Aug": "Precip_Jun_Aug_mm",
            "Sep_Nov": "Precip_Sep_Nov_mm",
            "Dec_Feb": "Precip_Dec_Feb_mm"
        })
    )

    rainfall_season = rainfall_year.merge(
        rainfall_periods,
        on="Season_Start_Year",
        how="left"
    )

    rainfall_season["Annual_Rainfall_mm"] = rainfall_season[
        "Annual_Rainfall_mm"
    ].round(3)

    print("Season-level rainfall table:")
    display(rainfall_season)
else:
    rainfall_season = pd.DataFrame()


Season-level rainfall table:


,Season_Start_Year,Annual_Rainfall_mm,Precip_Dec_Feb_mm,Precip_growth_phase_mm,Precip_Jun_Aug_mm,Precip_Sep_Nov_mm
0,2021,2211.56,42.03,420.24,1290.96,458.33
1,2022,1179.23,73.40,97.78,800.26,207.79
2,2023,576.31,2.46,54.49,373.04,146.32
3,2024,1494.68,7.82,83.85,900.75,502.26


In [10]:
# ============================================================
# 8. STAGING TRANSFORMATION
# ============================================================

staging = master.copy()

# Standardize key text fields.
text_columns = staging.select_dtypes(include="object").columns
for col in text_columns:
    staging[col] = staging[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

# Standardize Plot_ID.
staging["Plot_ID"] = staging["Plot_ID"].astype(str).str.strip()

# Ensure the main date fields are datetime.
for col in ["Planting_date", "Harvest_date"]:
    if col in staging.columns:
        staging[col] = pd.to_datetime(staging[col], errors="coerce")

# Convert the season to a consistent string representation.
if "Season_registry" in staging.columns:
    staging["Season"] = staging["Season_registry"].astype(str).str.strip()

staging["Season_Start_Year"] = staging["Season"].apply(season_start_year)

# Attach season-level DS18 rainfall.
if not rainfall_season.empty:
    staging = staging.merge(
        rainfall_season,
        on="Season_Start_Year",
        how="left"
    )

print("Staging shape:", staging.shape)
print("Staging unique plots:", staging["Plot_ID"].nunique())
display(staging[[
    "Plot_ID", "Season", "Season_Start_Year",
    "Annual_Rainfall_mm"
]].head())


Staging shape: (100, 132)
Staging unique plots: 100


,Plot_ID,Season,Season_Start_Year,Annual_Rainfall_mm
0,WC-001,2021-22,2021,2211.56
1,WC-002,2023-24,2023,576.31
2,WC-003,2023-24,2023,576.31
3,WC-004,2022-23,2022,1179.23
4,WC-005,2023-24,2023,576.31


## 9. Create Dimension Tables

Each dimension receives a surrogate integer key.

The dimensions intentionally contain descriptive attributes, while numerical measurements remain in the fact table.


In [11]:
# ============================================================
# 9A. DIM_PLOT
# ============================================================

plot_cols = [
    "Plot_ID",
    "Plot_Area_ha",
    "Irrigation_type",
    "Soil_type"
]

dim_plot = staging[plot_cols].drop_duplicates("Plot_ID").reset_index(drop=True)
dim_plot.insert(0, "Plot_Key", range(1, len(dim_plot) + 1))

print("Dim_Plot:", dim_plot.shape)
display(dim_plot.head())


Dim_Plot: (100, 5)


,Plot_Key,Plot_ID,Plot_Area_ha,Irrigation_type,Soil_type
0,1,WC-001,0.400,Canal,Sandy_loam
1,2,WC-002,1.435,Tubewell,Clay_loam
2,3,WC-003,1.550,Canal,Sandy_loam
3,4,WC-004,1.915,Canal,Alluvial
4,5,WC-005,1.193,Tubewell,Sandy_loam


In [12]:
# ============================================================
# 9B. DIM_LOCATION
# ============================================================

location_cols = [
    "Village_registry",
    "Block_registry",
    "Latitude_deg",
    "Longitude_deg"
]

dim_location = staging[location_cols].drop_duplicates().reset_index(drop=True)
dim_location.insert(0, "Location_Key", range(1, len(dim_location) + 1))

print("Dim_Location:", dim_location.shape)
display(dim_location.head())


Dim_Location: (100, 5)


,Location_Key,Village_registry,Block_registry,Latitude_deg,Longitude_deg
0,1,Matiaria,2,26.87983,84.67516
1,2,Sirsa,4,27.03752,84.74989
2,3,Matiaria,3,27.03765,84.74865
3,4,Matiaria,1,26.94336,84.57217
4,5,Sirsa,2,26.95796,84.65759


In [13]:
# ============================================================
# 9C. DIM_CROP
# ============================================================

crop_cols = ["Cultivar", "Cycle"]

dim_crop = staging[crop_cols].drop_duplicates().reset_index(drop=True)
dim_crop.insert(0, "Crop_Key", range(1, len(dim_crop) + 1))

print("Dim_Crop:", dim_crop.shape)
display(dim_crop.head())


Dim_Crop: (15, 3)


,Crop_Key,Cultivar,Cycle
0,1,CoVSI9,Plant
1,2,Co0238,Ratoon1
2,3,CoS767,Ratoon2
3,4,Co0238,Ratoon2
4,5,Co86032,Ratoon1


In [14]:
# ============================================================
# 9D. DIM_DATE
# ============================================================

# Use Planting_date as the principal agricultural observation date.
date_source = staging[["Planting_date", "Season"]].drop_duplicates().copy()
date_source = date_source.rename(columns={"Planting_date": "Date"})

dim_date = date_source.dropna(subset=["Date"]).drop_duplicates("Date").copy()
dim_date = dim_date.sort_values("Date").reset_index(drop=True)

dim_date["Day"] = dim_date["Date"].dt.day
dim_date["Month"] = dim_date["Date"].dt.month
dim_date["Month_Name"] = dim_date["Date"].dt.month_name()
dim_date["Quarter"] = "Q" + dim_date["Date"].dt.quarter.astype(str)
dim_date["Year"] = dim_date["Date"].dt.year

dim_date = dim_date[[
    "Date", "Day", "Month", "Month_Name", "Quarter", "Year"
]]
dim_date.insert(0, "Date_Key", range(1, len(dim_date) + 1))

print("Dim_Date:", dim_date.shape)
display(dim_date.head())


Dim_Date: (100, 7)


,Date_Key,Date,Day,Month,Month_Name,Quarter,Year
0,1,2021-11-01,1,11,November,Q4,2021
1,2,2021-11-04,4,11,November,Q4,2021
2,3,2021-11-07,7,11,November,Q4,2021
3,4,2021-11-10,10,11,November,Q4,2021
4,5,2021-11-13,13,11,November,Q4,2021


In [15]:
# ============================================================
# 9E. DIM_NUTRIENT
# ============================================================

nutrient_cols = ["N class", "P class", "K class"]

dim_nutrient = staging[nutrient_cols].drop_duplicates().reset_index(drop=True)
dim_nutrient.insert(0, "Nutrient_Key", range(1, len(dim_nutrient) + 1))

print("Dim_Nutrient:", dim_nutrient.shape)
display(dim_nutrient.head())


Dim_Nutrient: (14, 4)


,Nutrient_Key,N class,P class,K class
0,1,Excess,Adequate,Adequate
1,2,Deficient,Adequate,Adequate
2,3,Adequate,Excess,Adequate
3,4,Excess,Excess,Adequate
4,5,Adequate,Adequate,Adequate


In [16]:
# ============================================================
# 9F. DIM_VEGETATION
# ============================================================

vegetation_cols = [
    "GNDVI class",
    "Chl status class",
    "Growth stage"
]

dim_vegetation = staging[vegetation_cols].drop_duplicates().reset_index(drop=True)
dim_vegetation.insert(0, "Vegetation_Key", range(1, len(dim_vegetation) + 1))

print("Dim_Vegetation:", dim_vegetation.shape)
display(dim_vegetation.head())


Dim_Vegetation: (12, 4)


,Vegetation_Key,GNDVI class,Chl status class,Growth stage
0,1,High_vigour,High,Grand_growth
1,2,Low_vigour,Adequate,Tillering
2,3,High_vigour,High,Tillering
3,4,High_vigour,Adequate,Tillering
4,5,Moderate_vigour,High,Grand_growth


In [17]:
# ============================================================
# 9G. DIM_WEATHER
# ============================================================

weather_cols = [
    "Source",
    "Drought_flag",
    "Excess_rain_flag"
]

dim_weather = staging[weather_cols].drop_duplicates().reset_index(drop=True)
dim_weather.insert(0, "Weather_Key", range(1, len(dim_weather) + 1))

print("Dim_Weather:", dim_weather.shape)
display(dim_weather.head())


Dim_Weather: (5, 4)


,Weather_Key,Source,Drought_flag,Excess_rain_flag
0,1,ERA5_0.25deg,No,No
1,2,IMD_gridded,No,No
2,3,IMD_gridded,No,Yes
3,4,IMD_gridded,Yes,No
4,5,ERA5_0.25deg,Yes,No


## 10. Build the Fact Table

The fact table contains:
- foreign keys to the seven dimensions,
- yield and sugar-quality measures,
- rainfall measures,
- weather measures,
- nutrient measures,
- vegetation/chlorophyll measures.

The fact grain remains **one Plot × Season observation**.


In [18]:
# ============================================================
# 10.1 PREPARE DIMENSION LOOKUPS
# ============================================================

fact = staging.copy()

# Plot key
fact = fact.merge(
    dim_plot[["Plot_Key", "Plot_ID"]],
    on="Plot_ID",
    how="left"
)

# Location key
fact = fact.merge(
    dim_location.assign(_Location_Match=1),
    on=["Village_registry", "Block_registry", "Latitude_deg", "Longitude_deg"],
    how="left"
).drop(columns=["_Location_Match"], errors="ignore")

# Crop key
fact = fact.merge(
    dim_crop[["Crop_Key", "Cultivar", "Cycle"]],
    on=["Cultivar", "Cycle"],
    how="left"
)

# Date key
fact = fact.merge(
    dim_date[["Date_Key", "Date"]].rename(columns={"Date": "Planting_date"}),
    on="Planting_date",
    how="left"
)

# Nutrient key
fact = fact.merge(
    dim_nutrient[["Nutrient_Key"] + nutrient_cols],
    on=nutrient_cols,
    how="left"
)

# Vegetation key
fact = fact.merge(
    dim_vegetation[["Vegetation_Key"] + vegetation_cols],
    on=vegetation_cols,
    how="left"
)

# Weather key
fact = fact.merge(
    dim_weather[["Weather_Key"] + weather_cols],
    on=weather_cols,
    how="left"
)

print("Fact after key lookups:", fact.shape)


Fact after key lookups: (100, 139)


In [19]:
# ============================================================
# 10.2 SELECT FACT COLUMNS
# ============================================================

key_cols = [
    "Plot_Key",
    "Location_Key",
    "Crop_Key",
    "Date_Key",
    "Nutrient_Key",
    "Vegetation_Key",
    "Weather_Key"
]

measure_cols = [
    # Yield and quality
    "TCH_t_per_ha",
    "TSH_t_per_ha",
    "CCS_pct",
    "Sugar_recovery_pct",
    "Juice_purity_pct",
    "Brix_pct",
    "Pol_pct",

    # Existing weather
    "Precip_growth_phase_mm",
    "Precip_Jun_Aug_mm",
    "Precip_Sep_Nov_mm",
    "Precip_Dec_Feb_mm",
    "Annual_Rainfall_mm",
    "Dry_spell_days_gt5",
    "Wet_days_gt10mm",
    "Max_1day_precip_mm",
    "SPI_3month",

    # Nitrogen
    "N content pct DW",
    "Kjeldahl N mg kg",
    "Total protein pct",
    "N fertiliser kg ha",

    # Phosphorus
    "P content pct DW",
    "P mg per kg DW",
    "P2O5 pct",
    "Available P Olsen",
    "P fertiliser kg ha",
    "P uptake kg ha",

    # Potassium
    "K content pct DW",
    "K mg per kg DW",
    "K2O pct",
    "Exchangeable K meq",
    "K fertiliser kg ha",
    "K Na ratio",

    # Vegetation
    "GNDVI",
    "NDVI",
    "EVI",
    "LSWI",
    "Canopy LAI est",
    "Chlorophyll est ug cm2",

    # Chlorophyll
    "SPAD value",
    "Chl a mg per gFW",
    "Chl b mg per gFW",
    "Total Chl mg per gFW",
    "Chl a to b ratio",
    "Carotenoid mg per gFW",
    "Chl to Carotenoid ratio",
    "Fv Fm PSII efficiency",
    "Phi PSII quantum yield",
    "NPQ non photochem quenching",
    "Light use efficiency LUE"
]

# Keep only columns actually present.
available_measure_cols = [c for c in measure_cols if c in fact.columns]

fact_agriculture = fact[
    key_cols + available_measure_cols
].copy()

fact_agriculture.insert(
    0,
    "Agriculture_Fact_Key",
    range(1, len(fact_agriculture) + 1)
)

print("Fact_Agriculture shape:", fact_agriculture.shape)
display(fact_agriculture.head())


Fact_Agriculture shape: (100, 53)


,Agriculture_Fact_Key,Plot_Key,Location_Key,Crop_Key,Date_Key,Nutrient_Key,Vegetation_Key,Weather_Key,TCH_t_per_ha,TSH_t_per_ha,CCS_pct,Sugar_recovery_pct,Juice_purity_pct,Brix_pct,Pol_pct,Annual_Rainfall_mm,Dry_spell_days_gt5,Wet_days_gt10mm,Max_1day_precip_mm,SPI_3month,N content pct DW,Kjeldahl N mg kg,Total protein pct,N fertiliser kg ha,P content pct DW,P mg per kg DW,P2O5 pct,Available P Olsen,P fertiliser kg ha,P uptake kg ha,K content pct DW,K mg per kg DW,K2O pct,Exchangeable K meq,K fertiliser kg ha,K Na ratio,GNDVI,NDVI,EVI,LSWI,Canopy LAI est,Chlorophyll est ug cm2,SPAD value,Chl a mg per gFW,Chl b mg per gFW,Total Chl mg per gFW,Chl a to b ratio,Carotenoid mg per gFW,Chl to Carotenoid ratio,Fv Fm PSII efficiency,Phi PSII quantum yield,NPQ non photochem quenching,Light use efficiency LUE
0,1,1,1,1,1,1,1,1,11.18,1.023,11.80,7.71,84.18,19.36,20.64,2211.56,10,69,45.2,-0.465,2.896,28960.0,18.100,150.0,0.2241,2241.0,0.5134,19.56,50.0,13.44,1.482,14820.0,1.786,0.941,60.0,11.98,0.6253,0.8474,1.4089,0.1474,3.34,31.53,72.00,5.0000,1.5979,6.5979,3.129,1.3081,5.044,0.7545,0.6786,0.979,0.01773
1,2,2,2,2,2,2,1,2,104.19,9.136,11.42,7.76,86.65,20.42,20.91,576.31,4,76,168.5,0.612,1.499,14990.0,9.369,180.0,0.2157,2157.0,0.4942,30.41,40.0,9.77,1.290,12900.0,1.554,0.776,80.0,8.59,0.8444,0.7426,0.6684,0.6517,5.06,38.99,56.96,4.9785,2.0157,6.9942,2.470,1.5325,4.564,0.8500,0.8109,1.623,0.01752
2,3,3,3,3,3,3,2,1,71.81,6.720,10.75,7.83,86.83,21.26,16.78,576.31,4,44,134.2,1.638,1.616,16160.0,10.100,200.0,0.3566,3566.0,0.8170,51.26,40.0,25.94,1.843,18430.0,2.221,0.207,80.0,14.40,0.3347,0.7982,4.9820,0.0379,2.04,12.91,50.72,4.0485,1.3432,5.3917,3.014,1.1647,4.629,0.8499,0.7122,1.072,0.01477
3,4,4,4,4,4,3,3,2,51.50,4.583,11.93,8.24,92.49,22.75,15.22,1179.23,14,40,178.7,1.043,1.509,15090.0,9.431,200.0,0.3000,3000.0,0.6873,42.03,80.0,17.37,1.906,19060.0,2.297,0.848,100.0,10.71,0.7830,0.4611,0.3458,0.2131,4.58,35.75,56.01,4.9448,1.7080,6.6528,2.895,1.4622,4.550,0.7366,0.6689,1.409,0.01414
4,5,5,5,2,5,1,1,2,88.41,7.989,10.06,9.60,88.12,21.29,20.18,576.31,28,51,163.6,1.362,3.010,30100.0,18.812,200.0,0.2103,2103.0,0.4818,39.11,80.0,14.27,2.044,20440.0,2.463,0.657,60.0,13.03,0.6665,0.8019,1.3546,0.2645,3.60,27.91,72.00,5.0000,2.0529,7.0529,2.436,1.9679,3.584,0.8500,0.7704,2.660,0.02823


In [20]:
# ============================================================
# 11. ETL VALIDATION
# ============================================================

validation = {}

validation["Master rows"] = len(master)
validation["Fact rows"] = len(fact_agriculture)
validation["Master unique plots"] = master["Plot_ID"].nunique()
validation["Dimension Plot rows"] = len(dim_plot)
validation["Fact duplicate rows"] = fact_agriculture.duplicated().sum()

# Check missing foreign keys.
for key in key_cols:
    validation[f"Missing {key}"] = fact_agriculture[key].isna().sum()

validation_df = pd.DataFrame(
    list(validation.items()),
    columns=["Validation_Check", "Result"]
)

display(validation_df)

print("\nForeign-key validation:")
fk_missing = fact_agriculture[key_cols].isna().sum()
display(fk_missing.to_frame("Missing_Keys"))

assert fact_agriculture["Agriculture_Fact_Key"].is_unique
assert fact_agriculture["Plot_Key"].notna().all()
assert fact_agriculture["Crop_Key"].notna().all()
assert fact_agriculture["Nutrient_Key"].notna().all()
assert fact_agriculture["Vegetation_Key"].notna().all()
assert fact_agriculture["Weather_Key"].notna().all()

print("Core ETL validation checks passed.")


,Validation_Check,Result
0,Master rows,100
1,Fact rows,100
2,Master unique plots,100
3,Dimension Plot rows,100
4,Fact duplicate rows,0
5,Missing Plot_Key,0
6,Missing Location_Key,0
7,Missing Crop_Key,0
8,Missing Date_Key,0
9,Missing Nutrient_Key,0



Foreign-key validation:


,Missing_Keys
Plot_Key,0
Location_Key,0
Crop_Key,0
Date_Key,0
Nutrient_Key,0
Vegetation_Key,0
Weather_Key,0


Core ETL validation checks passed.


In [21]:
# ============================================================
# 11.1 GRAIN VALIDATION
# ============================================================

# Reconstruct the intended business grain from the staging data.
grain_check = staging[["Plot_ID", "Season"]].copy()

print("Rows at Plot × Season grain:", len(grain_check))
print("Duplicate Plot × Season combinations:",
      grain_check.duplicated(["Plot_ID", "Season"]).sum())

display(
    grain_check.groupby("Season")
    .agg(
        Plot_Count=("Plot_ID", "nunique"),
        Row_Count=("Plot_ID", "size")
    )
    .reset_index()
)

assert grain_check.duplicated(["Plot_ID", "Season"]).sum() == 0
print("Fact grain validated: one row per Plot × Season.")


Rows at Plot × Season grain: 100
Duplicate Plot × Season combinations: 0


,Season,Plot_Count,Row_Count
0,2021-22,40,40
1,2022-23,34,34
2,2023-24,26,26


Fact grain validated: one row per Plot × Season.


## 12. Save the Data Warehouse Tables

The warehouse is stored as separate CSV tables so the same tables can be:
- inspected independently,
- imported into Power BI,
- used to demonstrate dimensional modelling,
- versioned in GitHub,
- reused for further analytical work.


In [22]:
# ============================================================
# 12. SAVE WAREHOUSE TABLES
# ============================================================

tables = {
    "Dim_Plot": dim_plot,
    "Dim_Location": dim_location,
    "Dim_Crop": dim_crop,
    "Dim_Date": dim_date,
    "Dim_Nutrient": dim_nutrient,
    "Dim_Vegetation": dim_vegetation,
    "Dim_Weather": dim_weather,
    "Fact_Agriculture": fact_agriculture
}

for name, table in tables.items():
    path = WAREHOUSE_DIR / f"{name}.csv"
    table.to_csv(path, index=False)
    print(f"Saved {name}: {path} | shape={table.shape}")

validation_df.to_csv(
    WAREHOUSE_DIR / "ETL_Validation.csv",
    index=False
)

if not ds18.empty:
    rainfall_season.to_csv(
        WAREHOUSE_DIR / "DS18_Seasonal_Rainfall.csv",
        index=False
    )

print("\nWarehouse tables saved successfully.")


Saved Dim_Plot: C:\Users\Arindam Dwivedi\.vscode\sugarcane-agricultural-analytics\outputs\data_warehouse\Dim_Plot.csv | shape=(100, 5)
Saved Dim_Location: C:\Users\Arindam Dwivedi\.vscode\sugarcane-agricultural-analytics\outputs\data_warehouse\Dim_Location.csv | shape=(100, 5)
Saved Dim_Crop: C:\Users\Arindam Dwivedi\.vscode\sugarcane-agricultural-analytics\outputs\data_warehouse\Dim_Crop.csv | shape=(15, 3)
Saved Dim_Date: C:\Users\Arindam Dwivedi\.vscode\sugarcane-agricultural-analytics\outputs\data_warehouse\Dim_Date.csv | shape=(100, 7)
Saved Dim_Nutrient: C:\Users\Arindam Dwivedi\.vscode\sugarcane-agricultural-analytics\outputs\data_warehouse\Dim_Nutrient.csv | shape=(14, 4)
Saved Dim_Vegetation: C:\Users\Arindam Dwivedi\.vscode\sugarcane-agricultural-analytics\outputs\data_warehouse\Dim_Vegetation.csv | shape=(12, 4)
Saved Dim_Weather: C:\Users\Arindam Dwivedi\.vscode\sugarcane-agricultural-analytics\outputs\data_warehouse\Dim_Weather.csv | shape=(5, 4)
Saved Fact_Agriculture: C:

## 13. OLAP Operations

The following operations are demonstrated using pandas:

- **Roll-up:** aggregate Plot-level yield to Season.
- **Drill-down:** move from Season to Block/Village.
- **Slice:** select one Season.
- **Dice:** select a subset of Season + Nutrient class conditions.
- **Pivot:** reshape Season × Cultivar yield for comparison.

These operations represent the analytical behaviour expected from an OLAP/BI model.


In [24]:
# ============================================================
# 13.1 OLAP ROLL-UP
# Plot → Season
# ============================================================

# Season is already available in fact, so no merge is required.
olap_rollup = (
    fact.groupby("Season", as_index=False)
    .agg(
        Plot_Count=("Plot_ID", "nunique"),
        Average_TCH=("TCH_t_per_ha", "mean"),
        Average_TSH=("TSH_t_per_ha", "mean"),
        Average_CCS=("CCS_pct", "mean"),
        Average_Rainfall=("Annual_Rainfall_mm", "mean")
    )
    .round(3)
)

display(olap_rollup)

,Season,Plot_Count,Average_TCH,Average_TSH,Average_CCS,Average_Rainfall
0,2021-22,40,48.647,5.173,11.380,2211.56
1,2022-23,34,55.699,5.479,11.341,1179.23
2,2023-24,26,52.423,5.335,11.570,576.31


In [26]:
# ============================================================
# 13.2 OLAP DRILL-DOWN
# Season → Block
# ============================================================

# Season and Block are already available in the fact/staging data.
drilldown = (
    fact.groupby(
        ["Season", "Block_registry"],
        as_index=False
    )
    .agg(
        Plot_Count=("Plot_ID", "nunique"),
        Average_TCH=("TCH_t_per_ha", "mean"),
        Average_TSH=("TSH_t_per_ha", "mean")
    )
    .round(3)
)

display(drilldown.head(20))

,Season,Block_registry,Plot_Count,Average_TCH,Average_TSH
0,2021-22,1,10,63.403,6.757
1,2021-22,2,10,43.680,4.713
2,2021-22,3,9,67.672,7.214
3,2021-22,4,11,24.183,2.482
4,2022-23,1,6,57.870,5.324
5,2022-23,2,8,68.612,6.893
6,2022-23,3,10,58.475,5.792
7,2022-23,4,10,41.288,4.126
8,2023-24,1,5,63.188,6.127
9,2023-24,2,11,50.650,5.207


In [27]:
# ============================================================
# 13.3 OLAP SLICE
# Select one season
# ============================================================

available_seasons = sorted(staging["Season"].dropna().unique())

if available_seasons:
    selected_season = available_seasons[0]
    slice_df = staging[staging["Season"] == selected_season].copy()

    print("Selected season:", selected_season)
    print("Rows:", len(slice_df))
    display(
        slice_df[[
            "Plot_ID", "Season", "Cultivar",
            "TCH_t_per_ha", "TSH_t_per_ha"
        ]].head(10)
    )


Selected season: 2021-22
Rows: 40


,Plot_ID,Season,Cultivar,TCH_t_per_ha,TSH_t_per_ha
0,WC-001,2021-22,CoVSI9,11.18,1.023
5,WC-006,2021-22,Co86032,136.25,15.845
7,WC-008,2021-22,CoVSI9,29.13,3.383
8,WC-009,2021-22,Co0238,49.26,5.510
10,WC-011,2021-22,CoVSI9,7.42,0.750
12,WC-013,2021-22,Co0238,47.27,5.425
16,WC-017,2021-22,CoS767,11.46,1.086
19,WC-020,2021-22,Co86032,49.20,5.975
21,WC-022,2021-22,CoVSI9,84.45,9.282
24,WC-025,2021-22,CoVSI9,57.19,5.853


In [28]:
# ============================================================
# 13.4 OLAP DICE
# Filter by season + nutrient class
# ============================================================

dice_cols = [
    "Plot_ID", "Season", "Cultivar",
    "N class", "P class", "K class",
    "TCH_t_per_ha", "TSH_t_per_ha"
]

dice_source = staging[dice_cols].copy()

# Example: retain rows with a non-missing N class and TCH.
dice_df = dice_source[
    dice_source["N class"].notna() &
    dice_source["TCH_t_per_ha"].notna()
].copy()

display(dice_df.head(10))


,Plot_ID,Season,Cultivar,N class,P class,K class,TCH_t_per_ha,TSH_t_per_ha
0,WC-001,2021-22,CoVSI9,Excess,Adequate,Adequate,11.18,1.023
1,WC-002,2023-24,Co0238,Deficient,Adequate,Adequate,104.19,9.136
2,WC-003,2023-24,CoS767,Adequate,Excess,Adequate,71.81,6.720
3,WC-004,2022-23,Co0238,Adequate,Excess,Adequate,51.50,4.583
4,WC-005,2023-24,Co0238,Excess,Adequate,Adequate,88.41,7.989
5,WC-006,2021-22,Co86032,Deficient,Adequate,Adequate,136.25,15.845
6,WC-007,2022-23,Co0238,Excess,Excess,Adequate,17.69,1.824
7,WC-008,2021-22,CoVSI9,Adequate,Adequate,Adequate,29.13,3.383
8,WC-009,2021-22,Co0238,Adequate,Adequate,Adequate,49.26,5.510
9,WC-010,2022-23,Co86032,Adequate,Adequate,Adequate,135.07,11.769


In [29]:
# ============================================================
# 13.5 OLAP PIVOT
# Season × Cultivar → Average TCH
# ============================================================

olap_pivot = pd.pivot_table(
    staging,
    index="Season",
    columns="Cultivar",
    values="TCH_t_per_ha",
    aggfunc="mean"
).round(3)

display(olap_pivot)


Cultivar,Co0238,Co86032,CoJ64,CoS767,CoVSI9
Season,,,,,
2021-22,68.552,33.064,57.267,43.868,62.536
2022-23,43.487,59.447,51.666,81.878,49.805
2023-24,55.284,38.412,75.790,48.421,63.790


## 14. Analytical KPI Layer

These measures can be recreated directly in Power BI from `Fact_Agriculture`.


In [30]:
# ============================================================
# 14. KPI SUMMARY
# ============================================================

kpi_summary = pd.DataFrame({
    "KPI": [
        "Average TCH (t/ha)",
        "Average TSH (t/ha)",
        "Average CCS (%)",
        "Average Sugar Recovery (%)",
        "Average Annual Rainfall (mm)",
        "Maximum TCH (t/ha)",
        "Minimum TCH (t/ha)"
    ],
    "Value": [
        fact_agriculture["TCH_t_per_ha"].mean(),
        fact_agriculture["TSH_t_per_ha"].mean(),
        fact_agriculture["CCS_pct"].mean(),
        fact_agriculture["Sugar_recovery_pct"].mean(),
        fact_agriculture["Annual_Rainfall_mm"].mean(),
        fact_agriculture["TCH_t_per_ha"].max(),
        fact_agriculture["TCH_t_per_ha"].min()
    ]
}).round(3)

display(kpi_summary)


,KPI,Value
0,Average TCH (t/ha),52.027
1,Average TSH (t/ha),5.319
2,Average CCS (%),11.416
3,Average Sugar Recovery (%),9.178
4,Average Annual Rainfall (mm),1435.403
5,Maximum TCH (t/ha),218.390
6,Minimum TCH (t/ha),2.290


## 15. Power BI Loading Plan

For the MDA presentation, import the eight warehouse CSV files from:

`outputs/data_warehouse/`

Recommended Power BI relationships:

```text
Dim_Plot       1 ─── * Fact_Agriculture
Dim_Location   1 ─── * Fact_Agriculture
Dim_Crop       1 ─── * Fact_Agriculture
Dim_Date       1 ─── * Fact_Agriculture
Dim_Nutrient   1 ─── * Fact_Agriculture
Dim_Vegetation 1 ─── * Fact_Agriculture
Dim_Weather    1 ─── * Fact_Agriculture
```

Use `Fact_Agriculture` for numerical measures and dimensions for slicing/filtering.

### Suggested OLAP visuals

- Season-level yield → **Roll-up**
- Season → Block → Plot → **Drill-down**
- One selected season → **Slice**
- Season + N/P/K class filters → **Dice**
- Season × Cultivar matrix → **Pivot**


## 16. Final MDA Task Mapping

| MDA Task | Evidence in Project |
|---|---|
| 1. Business Requirements | KPI and analytical requirements |
| 2. Data Sources | 8 Excel datasets + DS18 rainfall |
| 3. Analyze Source Data | Notebook 1 |
| 4. Warehouse Architecture | Source → Staging → Warehouse → OLAP/BI |
| 5. Dimensional Model | Star Schema + fact grain |
| 6. Extract Source Data | Master Dataset + DS18 loading |
| 7. Data Cleaning | Reuses validated cleaning from Notebooks 1–2 |
| 8. Data Transformation | Standardization + seasonal rainfall transformation |
| 9. Dimension Tables | Seven dimension CSVs |
| 10. Fact Table | `Fact_Agriculture.csv` |
| 11. ETL Validation | Grain, row count and foreign-key checks |
| 12. Analytical Model | Star Schema / Power BI model |
| 13. OLAP Operations | Roll-up, Drill-down, Slice, Dice, Pivot |
| 14. Interactive Dashboard | Power BI using warehouse tables |
| 15. Test, Document & Present | Validation + documentation + PPT/video |

### Important distinction

Notebook 1 and Notebook 2 remain the **source preparation and analytical layer**.

This Notebook 3 is the **MDA / Data Warehouse / ETL / OLAP layer**. It therefore adds architecture and dimensional modelling instead of repeating the previous EDA.


## 17. Final Conclusion

The cleaned sugarcane Master Dataset has been converted into a dimensional analytical structure using a **Plot × Season fact grain**.

The ETL workflow:
1. extracts the validated Master Dataset and DS18 rainfall,
2. stages and standardizes the data,
3. transforms monthly rainfall to season-level measures,
4. creates surrogate-key dimensions,
5. loads the agriculture fact table,
6. validates keys and grain,
7. saves warehouse tables,
8. demonstrates OLAP operations,
9. prepares the model for Power BI.

This completes the MDA-specific data warehouse and OLAP component of the sugarcane analytics project.
